# End-to-End ETL Pipeline (Local)

Runs the full WeatherSpeak PH pipeline locally using `modal_etl/core/` modules.

**Pipeline:** PDF → OCR markdown + metadata → radio scripts + TTS text → MP3

The same `run_step1 / run_step2 / run_step3` functions are used by the Modal ETL in production.
Output mirrors the Modal Volume layout: `output/{stem}/ocr.md`, `radio_{lang}.md`, etc.

Set `force=True` in any step cell to re-run that step even if outputs already exist.

In [1]:
import sys
from pathlib import Path

# Make modal_etl importable from notebook directory
sys.path.insert(0, str(Path.cwd().parent))

from modal_etl.core.ocr import run_step1
from modal_etl.core.scripts import run_step2
from modal_etl.core.tts import run_step3

In [2]:
# ── Configuration ────────────────────────────────────────────────────────
STEM          = "PAGASA_25-TC22_Verbena_TCB#24"
PDF_PATH      = Path("../data/bulletin-archive/archive/pagasa-25-TC22") / f"{STEM}.pdf"
OUTPUT_DIR    = Path("10-etl-e2e/output")
OLLAMA_URL    = "http://localhost:11434"
LANGUAGES     = ["ceb", "tl", "en"]
TTS_MODELS_DIR = Path.home() / ".cache" / "huggingface" / "hub"
FORCE_RUN       = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"PDF:        {PDF_PATH}  (exists={PDF_PATH.exists()})")
print(f"Output dir: {OUTPUT_DIR.resolve()}")
print(f"Ollama URL: {OLLAMA_URL}")

PDF:        ../data/bulletin-archive/archive/pagasa-25-TC22/PAGASA_25-TC22_Verbena_TCB#24.pdf  (exists=True)
Output dir: /Users/josereyes/Dev/gemma4-hackathon/notebooks/10-etl-e2e/output
Ollama URL: http://localhost:11434


## Step 1 Validation

Runs OCR + metadata for the bulletin configured in `STEM` above, then checks outputs.

In [3]:
# (run the Step 1 cell below to execute OCR for the configured STEM)

In [4]:
import json

stem_dir = OUTPUT_DIR / STEM

# Check ocr.md does NOT contain forecast table rows
ocr_md = (stem_dir / "ocr.md").read_text(encoding="utf-8")
table_leaked = any(f"{h}-Hour Forecast" in ocr_md for h in [12, 24, 36, 48, 60, 72, 96, 120])
print(f"ocr.md: {len(ocr_md)} chars | forecast table leaked: {table_leaked}")

# Check forecast_table.md
ft_md = (stem_dir / "forecast_table.md").read_text(encoding="utf-8")
print(f"forecast_table.md: {len(ft_md)} chars")

# Check new fields in metadata.json
meta = json.loads((stem_dir / "metadata.json").read_text(encoding="utf-8"))
print(f"wind_extent:   {meta.get('wind_extent')}")
print(f"land_hazards:  {meta.get('land_hazards')}")
print(f"track_outlook: {meta.get('track_outlook')}")
print(f"forecast_positions: {len(meta.get('forecast_positions', []))} rows")


## Step 1: OCR — PDF → Markdown + Metadata

Sends each PDF page to Gemma 4 E4B via Ollama vision API.
Writes `ocr.md`, `chart.png`, and `metadata.json` to `output/{stem}/`.

In [5]:
stem_dir = run_step1(PDF_PATH, OUTPUT_DIR, ollama_url=OLLAMA_URL, force=FORCE_RUN)
print(f"\nStep 1 complete → {stem_dir}")

# Preview OCR markdown
ocr_md = (stem_dir / "ocr.md").read_text(encoding="utf-8")
print(f"\n--- ocr.md preview (first 500 chars) ---")
print(ocr_md[:500])

# Pretty-print metadata
import json
metadata = json.loads((stem_dir / "metadata.json").read_text(encoding="utf-8"))
print(f"\n--- metadata.json ---")
print(json.dumps(metadata, indent=2, ensure_ascii=False)[:1000])

[run_step1] PAGASA_25-TC22_Verbena_TCB#24: wrote ocr.md (8104 chars)
[run_step1] PAGASA_25-TC22_Verbena_TCB#24: wrote forecast_table.md (1351 chars)
[run_step1] PAGASA_25-TC22_Verbena_TCB#24: saved chart.png (page 0)
[run_step1] PAGASA_25-TC22_Verbena_TCB#24: wrote metadata.json

Step 1 complete → 10-etl-e2e/output/PAGASA_25-TC22_Verbena_TCB#24

--- ocr.md preview (first 500 chars) ---
<!-- Page 1 -->

**Bulletin type and number:** TROPICAL CYCLONE BULLETIN NR. 24F

**Storm current name, former name (if any), international name (if any):** VERBENA

**Issue date and time:** 11:00 AM, 27 November 2025

**Headline:** "VERBENA" WEAKENS WHILE MOVING WEST SOUTHWESTWARD SLOWLY

**Location of Center:** 10.00 PM (Coordinates not explicitly listed for this section, but the map shows the general area)

**Intensity:**
*   Max sustained winds: 120 km/h
*   Gusts: up to 150 km/h
*   Central

--- metadata.json ---
{
  "bulletin_type": "TCA",
  "storm": {
    "name": "VERBENA",
    "category": "Tropic

## Step 2: Radio Scripts + TTS Text

Generates a spoken weather announcement and TTS-optimised plain text for each language.
Writes `radio_{lang}.md` and `tts_{lang}.txt` to `output/{stem}/`.

In [6]:
for lang in LANGUAGES:
    radio_path = run_step2(STEM, lang, OUTPUT_DIR, ollama_url=OLLAMA_URL, force=FORCE_RUN)
    print(f"\n{'='*60}")
    print(f"[{lang.upper()}] {radio_path.name}")
    print('='*60)
    print(radio_path.read_text(encoding="utf-8")[:400])
print("\nStep 2 complete.")

[run_step2] PAGASA_25-TC22_Verbena_TCB#24/ceb: using hybrid input (metadata + OCR)
[run_step2] PAGASA_25-TC22_Verbena_TCB#24/ceb: wrote radio + tts files

[CEB] radio_ceb.md
Ang bagyo kay Tropical Storm VERBENA. Kasamtangang kusog pa kaayo ang hangin, pero hinay ra siyang naglihok paingon sa baratong habagatan. Ang pag-ilis sa kuryente ug tubig, pag-andam sa paglikas mao ang importante karon.

Adunay Signal 2 nga maoy gibati sa Bataan, Cagayan, Ilocos Norte, Ilocos Sur, ug Palawan. Ang mga rehiyon sa Visayas ug Mindanao nagpabiling nagbantay pa gihapon.

Para sa tanan
[run_step2] PAGASA_25-TC22_Verbena_TCB#24/tl: using hybrid input (metadata + OCR)
[run_step2] PAGASA_25-TC22_Verbena_TCB#24/tl: wrote radio + tts files

[TL] radio_tl.md
Bagyo na VERBENA. Malakas pa ito ngayon, may hangin na umabot sa 120 kilometro bawat oras. Galaw nito ay pa-kanluran-timog-kanluran.

Ang mga lugar na babantayan ay ang Bataan, Cagayan, Apario, Ilocos Norte, Ilocos Sur, at Palawan, na may Signal 2. May ba

## Step 3: TTS Synthesis → MP3

Synthesizes MP3 audio for each language using:
- English: Coqui XTTS v2 (speaker: Damien Black)
- Tagalog / Cebuano: Facebook MMS VITS

Writes `audio_{lang}.mp3` to `output/{stem}/`.

In [7]:
for lang in LANGUAGES:
    mp3_path = run_step3(STEM, lang, OUTPUT_DIR, TTS_MODELS_DIR, force=FORCE_RUN)
    size_kb = mp3_path.stat().st_size // 1024
    print(f"[{lang.upper()}] {mp3_path.name}  ({size_kb} KB)")
print("\nStep 3 complete.")

/Users/josereyes/Dev/gemma4-hackathon/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[MMSSynthesizer] loading facebook/mms-tts-ceb on cpu
[run_step3] PAGASA_25-TC22_Verbena_TCB#24/ceb: wrote 10-etl-e2e/output/PAGASA_25-TC22_Verbena_TCB#24/audio_ceb.mp3
[CEB] audio_ceb.mp3  (901 KB)
[MMSSynthesizer] loading facebook/mms-tts-tgl on cpu
[run_step3] PAGASA_25-TC22_Verbena_TCB#24/tl: wrote 10-etl-e2e/output/PAGASA_25-TC22_Verbena_TCB#24/audio_tl.mp3
[TL] audio_tl.mp3  (843 KB)
[CoquiXTTSSynthesizer] loading on cpu
[run_step3] PAGASA_25-TC22_Verbena_TCB#24/en: wrote 10-etl-e2e/output/PAGASA_25-TC22_Verbena_TCB#24/audio_en.mp3
[EN] audio_en.mp3  (726 KB)

Step 3 complete.


In [8]:
from IPython.display import Audio, display, Markdown

for lang in LANGUAGES:
    mp3_path = OUTPUT_DIR / STEM / f"audio_{lang}.mp3"
    if mp3_path.exists():
        display(Markdown(f"**{lang.upper()}**"))
        display(Audio(str(mp3_path)))

**CEB**

**TL**

**EN**

## Output Summary

In [9]:
stem_dir = OUTPUT_DIR / STEM
print(f"Artefacts in {stem_dir.resolve()}:\n")
for f in sorted(stem_dir.iterdir()):
    size = f.stat().st_size
    unit = 'KB' if size >= 1024 else 'B'
    val = size // 1024 if size >= 1024 else size
    print(f"  {f.name:<35}  {val:>6} {unit}")

Artefacts in /Users/josereyes/Dev/gemma4-hackathon/notebooks/10-etl-e2e/output/PAGASA_25-TC22_Verbena_TCB#24:

  audio_ceb.mp3                           901 KB
  audio_en.mp3                            726 KB
  audio_tl.mp3                            843 KB
  chart.png                               687 KB
  forecast_table.md                         1 KB
  metadata.json                             2 KB
  ocr.md                                    7 KB
  radio_ceb.md                            638 B
  radio_en.md                             502 B
  radio_tl.md                             760 B
  tts_ceb.txt                             645 B
  tts_en.txt                              516 B
  tts_tl.txt                              821 B
